In [1]:
import importlib as imp
import os
import random
import sys
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_probability as tfp

sys.path.append("../")
import build_data
import build_model
import compute_metrics
import compute_predictions
import custom_loss
import experiment_settings as exps
import model_diagnostics
import train_experiments

import martracks

%matplotlib inline
%load_ext autotime
warnings.filterwarnings("ignore")

time: 104 µs (started: 2023-06-06 13:10:25 -06:00)


In [2]:
# pipeline for testing:
### 1. make/add experiment -- new_exp = exps.Experiment().make_exp_dictionary(settings) --> exps.Experiment().add_experiment(new_exp)
##### a) manually add one each for AL/EP (2 models), set testing_years=None to go through each year (10 models), leadtimes=None does every 24 hours (5 models)
### 2. train experiment -- exps.Experiment().run_experiments(exp_name, model_path, metric_path, prediction_path)
##### a) this saves the default metrics and makes some figures (training/validation history) -- use unique paths for organization
### 3. compute and save my metrics -- my_metrics = martrics.Martrics(path/to/saved/predictions, path/to/saved/metrics/metric_savefile_name)
### 4. make/save my plots -- my_metrics.plot_all(figure/path)
### 5. make/save Libby plots --  martrics.plot_metric_comparison(metric_path, figure_path)
imp.reload(exps)
imp.reload(custom_loss)
imp.reload(build_data)
imp.reload(build_model)
imp.reload(train_experiments)
imp.reload(compute_predictions)
imp.reload(compute_metrics)
imp.reload(model_diagnostics)

testing = exps.Experiments()

AttributeError: module 'experiment_settings' has no attribute 'Experiments'

time: 360 ms (started: 2023-06-06 13:10:25 -06:00)


In [ ]:
# testing.restore_from_backup("experiments-backup.json", confirm=1)
# testing.save_backup("experiments-backup.json")

In [4]:
# testing.delete_experiment(testing.get_exp_list[-1])
testing.get_exp_list

['centered_bivariate_normal_000_EP',
 'centered_bivariate_normal_AL',
 'centered_bivariate_normal_EP',
 'single_NN12_val1600_AL',
 'single_NN12_val1600_EP',
 'single_NN12_eqval_AL',
 'single_NN12_eqval_EP',
 'single_NN12_2023_AL',
 'single_NN12_2023_EP',
 'single_NN12_l15l10_AL',
 'single_NN12_l15l10_EP',
 'single_NN12_l5l5l5_AL',
 'single_NN12_l5l5l5_EP',
 'single_NN12_batch32_AL',
 'single_NN12_batch32_EP',
 'single_NN12_batch96_AL',
 'single_NN12_batch96_EP',
 'single_NN12_removeNCT_AL',
 'single_NN12_removeNCT_EP',
 'single_NN12_removeVMAX_AL',
 'single_NN12_removeVMAX_EP',
 'single_NN12_removeHWRF_AL',
 'single_NN12_removeHWRF_EP',
 'single_NN12_addLATLON0_AL',
 'single_NN12_addLATLON0_EP',
 'single_NN12_addLATLONN_AL',
 'single_NN12_addLATLONN_EP',
 'single_NN12_removeDV_AL',
 'single_NN12_removeDV_EP',
 'single_NN12_addD200_AL',
 'single_NN12_addD200_EP',
 'single_NN12_locinp_AL',
 'single_NN12_locinp_EP',
 'single_NN12_noHWRF_AL',
 'single_NN12_noHWRF_EP',
 'single_NN12_losswei

time: 1.22 ms (started: 2023-06-05 14:13:58 -06:00)


In [5]:
# - leadtimes: multiples of 12 up to 168 hours, or None to generate separate experiments for each leadtime, 0 to generate a network that uses all leadtimes
# - x_names: features to use -- entering x_names that are already in default_x_names removes them, otherwise they are added
# - predictand: the label, either 'OFDV' (official forecast error) or 'OBDV' (consensus error)
# - shashn: the number of parameters to use for the SHASH, 'shash2', 'shash3', or 'shash4'
# - undersample: whether to undersample
# - hiddens: number of nodes in each layer (length of list determines number of layers)
# - dropout: fraction for dropout in each layer
# - ridge: ridge parameter for each layer
# - learning: learning rate
# - momentum: ???
# - nest: ???
# - batch: batch size
# - seed_list: seeds to use for training/testing runs
# - act_fun: activation function, 'relu' as default. Other options available through tensorflow
# - patience: number of unimproved training epochs before returning best epoch
# - n_val: number of validation samples to use, if <= 1, uses a fraction of the training samples
# - loss_function: the loss function to use. See custom_loss.py for options
# - metrics: metrics to use for validation, should be a dictionary of {function: name}. See custom_metrics.py for options
# - normalize: whether to normalize the input features
# - add_disp: add uniform random displacement to labels
# - basin_aug: augment the validation set with data from the other basin

time: 331 µs (started: 2023-06-05 14:13:58 -06:00)


In [48]:
### 1.
# al_exp = testing.make_exp_dictionary(expname='single_NN12_swaplocation_AL', basin="AL", years_test=None, x_names=['LON0', 'LAT0', 'LONC', 'LATC'])
# ep_exp = testing.make_exp_dictionary(expname='single_NN12_swaplocation_EP', basin="EP", years_test=None, x_names=['LON0', 'LAT0', 'LONC', 'LATC'])

al_exp = testing.make_exp_dictionary(expname='single_NN12_sigmoid_AL', basin="AL", years_test=None, act_fun="sigmoid")
ep_exp = testing.make_exp_dictionary(expname='single_NN12_sigmoid_EP', basin="EP", years_test=None, act_fun="sigmoid")

using features:  ['ftime(hr)', 'NCT', 'VMAX0', 'AVDX', 'EMDX', 'EGDX', 'HWDX', 'AVDY', 'EMDY', 'EGDY', 'HWDY', 'LONC', 'LATC', 'VMXC', 'DV12', 'SLAT', 'SHDC', 'SSTN', 'DTL', 'DSDV', 'LGDV', 'HWDV', 'AVDV', 'SPDX', 'SPDY']
using features:  ['ftime(hr)', 'NCT', 'VMAX0', 'AVDX', 'EMDX', 'EGDX', 'HWDX', 'AVDY', 'EMDY', 'EGDY', 'HWDY', 'LONC', 'LATC', 'VMXC', 'DV12', 'SLAT', 'SHDC', 'SSTN', 'DTL', 'DSDV', 'LGDV', 'HWDV', 'AVDV', 'SPDX', 'SPDY']
time: 331 µs (started: 2023-06-06 08:15:51 -06:00)


In [49]:
testing.add_experiment(al_exp)
testing.add_experiment(ep_exp)
testing.get_exp_list_short

array(['centered_bivariate_normal', 'centered_bivariate_normal_000',
       'single_NN12_2023', 'single_NN12_addD200',
       'single_NN12_addLATLON0', 'single_NN12_addLATLONN',
       'single_NN12_batch32', 'single_NN12_batch96',
       'single_NN12_dropout0.2', 'single_NN12_dropout0.4',
       'single_NN12_elu', 'single_NN12_eqval', 'single_NN12_gelu',
       'single_NN12_l15l10', 'single_NN12_l15l10l5', 'single_NN12_l25l5',
       'single_NN12_l2o', 'single_NN12_l5l5l5', 'single_NN12_locinp',
       'single_NN12_lossweights', 'single_NN12_no120',
       'single_NN12_noHWRF', 'single_NN12_nocirc',
       'single_NN12_nolocation', 'single_NN12_removeDV',
       'single_NN12_removeHWRF', 'single_NN12_removeNCT',
       'single_NN12_removeVMAX', 'single_NN12_selu',
       'single_NN12_sigmoid', 'single_NN12_swaplocation',
       'single_NN12_undersample', 'single_NN12_val1600',
       'single_NN12_weights^2_noeqval'], dtype='<U29')

time: 3.68 ms (started: 2023-06-06 08:15:51 -06:00)


In [50]:
### 2.
for test in testing.get_exp_list:
    if test.startswith('single_NN12_sigmoid'):
        main_exp = test[::-1].split('_', maxsplit=1)[1][::-1]
        testing.run_experiments(test, 
                                model_path=os.path.join('../track_martin/saved_models/', main_exp+'/'), 
                                metric_path=os.path.join('../track_martin/saved_metrics/', main_exp+'/'),
                                prediction_path=os.path.join('../track_martin/saved_predictions/', main_exp+'/'),
                                overwrite_predictions=True, overwrite_model=True)

Training single_NN12_sigmoid_AL_2013_centered_bivariate_normal_rng_seed_123_OF
Restoring model weights from the end of the best epoch: 10380.
Epoch 10630: early stopping
{'best_epoch': 10379,
 'elapsed_time': 935.4796369075775,
 'loss_train': 11.352402687072754,
 'loss_valid': 11.352668762207031}
Training single_NN12_sigmoid_AL_2014_centered_bivariate_normal_rng_seed_123_OF
Restoring model weights from the end of the best epoch: 4068.
Epoch 04318: early stopping
{'best_epoch': 4067,
 'elapsed_time': 365.66069078445435,
 'loss_train': 11.380391120910645,
 'loss_valid': 11.392685890197754}
Training single_NN12_sigmoid_AL_2015_centered_bivariate_normal_rng_seed_123_OF
Restoring model weights from the end of the best epoch: 13595.
Epoch 13845: early stopping
{'best_epoch': 13594,
 'elapsed_time': 1165.23886013031,
 'loss_train': 11.343944549560547,
 'loss_valid': 11.374446868896484}
Training single_NN12_sigmoid_AL_2016_centered_bivariate_normal_rng_seed_123_OF
Restoring model weights from 

In [3]:
main_exp = 'default_noHWRF'

time: 156 µs (started: 2023-06-06 16:21:11 -06:00)


In [4]:
### 3.
imp.reload(martracks)
metric_savefile_name = "martrics"
my_metrics = martracks.Martrics(os.path.join('../track_martin/saved_predictions/', main_exp), os.path.join('../track_martin/saved_metrics/', main_exp, metric_savefile_name))

time: 10min 42s (started: 2023-06-06 16:21:12 -06:00)


In [5]:
### 4.
# imp.reload(martracks)
my_metrics.plot_all("../track_martin/figures/"+main_exp)

time: 25.5 s (started: 2023-06-06 16:31:54 -06:00)


In [6]:
### 5.
# imp.reload(martracks)
martracks.plot_metric_comparison(os.path.join('../track_martin/saved_metrics/', main_exp), main_exp, "../track_martin/figures/"+main_exp)
martracks.plot_metric_sidebyside(os.path.join('../track_martin/saved_metrics/', main_exp), main_exp, "../track_martin/figures/"+main_exp)

time: 4 s (started: 2023-06-06 16:32:20 -06:00)


In [5]:
# martracks.plot_metric_sidebyside(os.path.join('../track_martin/saved_metrics/', main_exp), main_exp, "../track_martin/figures/"+main_exp, default_metric_path='../track_martin/saved_metrics/single_NN12_2023/', default_exp='single_NN12_2023')

time: 1.58 s (started: 2023-06-03 16:41:32 -06:00)


In [12]:
# # some input parameter tests
# # par_tests = np.array([["NCT"], ["VMAX0", "VMXC"], ["HWDX", "HWDY"], ["LAT0", "LON0"], ["LATN", "LONN"], ["DSDV", "AVDV", "LGDV", "HWDV"], ["D200"]])
# # par_names = np.array(["removeNCT", "removeVMAX", "removeHWRF", "addLATLON0", "addLATLONN", "removeDV", "addD200"])
# par_tests = np.array([["HWDX", "HWDY"], ["LAT0", "LON0"], ["LATN", "LONN"], ["DSDV", "AVDV", "LGDV", "HWDV"], ["D200"]])
# par_names = np.array(["removeHWRF", "addLATLON0", "addLATLONN", "removeDV", "addD200"])

# for ii in range(par_names.size):

#     al_exp = testing.make_exp_dictionary(expname='single_NN12_'+par_names[ii]+'_AL', basin="AL", years_test=None, x_names=par_tests[ii])
#     ep_exp = testing.make_exp_dictionary(expname='single_NN12_'+par_names[ii]+'_EP', basin="EP", years_test=None, x_names=par_tests[ii])

#     testing.add_experiment(al_exp)
#     testing.add_experiment(ep_exp)
    
#     metric_savefile_name = "martrics"

#     for test in testing.get_exp_list:
#         if test.startswith('single_NN12_'+par_names[ii]):
#             main_exp = test[::-1].split('_', maxsplit=1)[1][::-1]
#             testing.run_experiments(test, 
#                                     model_path=os.path.join('../track_martin/saved_models/', main_exp+'/'), 
#                                     metric_path=os.path.join('../track_martin/saved_metrics/', main_exp+'/'),
#                                     prediction_path=os.path.join('../track_martin/saved_predictions/', main_exp+'/'),
#                                     overwrite_predictions=True, overwrite_model=True)

#     my_metrics = martracks.Martrics(os.path.join('../track_martin/saved_predictions/', main_exp), os.path.join('../track_martin/saved_metrics/', main_exp, metric_savefile_name))

#     my_metrics.plot_all("../track_martin/figures/"+main_exp)

#     martracks.plot_metric_comparison(os.path.join('../track_martin/saved_metrics/', main_exp), main_exp, "../track_martin/figures/"+main_exp)

time: 448 µs (started: 2023-05-17 10:42:32 -06:00)
